# Maternal Health Risk Prediction & Early Warning System

**Pipeline:** Dataset → Data cleaning → EDA → Preprocessing → Train / Validation / Test split → Class-imbalance check → Baseline models → Hyperparameter tuning (hold-out validation set, **no cross-validation**) → Final training → Evaluation (Accuracy, Precision, Recall, F1, Sensitivity, Specificity, ROC-AUC, Confusion Matrix) → Best model → XAI comparison (SHAP vs LIME vs Permutation Importance) → Predictive system (with abnormal-feature report) → Save model

Put `Maternal_Health_Risk_RAW_UNCLEANED.csv` in the same folder as this notebook, then run the cells top to bottom.

**Setup**  (needs `pip install lime shap imbalanced-learn plotly`)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle, FancyBboxPatch
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                              classification_report, confusion_matrix, roc_auc_score, roc_curve)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import shap

try:
    get_ipython().run_line_magic("config", "InlineBackend.print_figure_kwargs={'bbox_inches': None}")
except NameError:
    pass

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42
LABELS = ["Low", "Mid", "High"]
COLORS = ["#2E9E6D", "#F0A500", "#E4572E"]
RISK_COLORS = dict(zip(["low risk", "mid risk", "high risk"], COLORS))
ORDER = ["low risk", "mid risk", "high risk"]

## 1. Conceptual Model & Pipeline Diagrams

**1a. Circular Conceptual Model**

In [ ]:
stages = [
    "Dataset\n(14 clinical\nfeatures)", "EDA", "Preprocessing", "Train/Test\nSplit",
    "Scaling +\nSMOTE", "Model\nTraining", "Hyperparameter\nTuning (Val set)", "Evaluation",
    "Best Model\nSelection", "SHAP\nExplainability", "Risk Prediction\n(Low/Mid/High)", "Early Warning\nAlert",
]
n = len(stages)
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"aspect": "equal"})
R = 4.6
angles = [np.pi/2 - 2*np.pi*i/n for i in range(n)]
node_colors = plt.cm.viridis(np.linspace(0.08, 0.92, n))

for i, (stage, ang) in enumerate(zip(stages, angles)):
    x, y = R*np.cos(ang), R*np.sin(ang)
    ax.add_patch(Circle((x, y), 0.92, facecolor=node_colors[i], edgecolor="white", linewidth=2.5, zorder=3))
    ax.text(x, y, stage, ha="center", va="center", fontsize=9.3, fontweight="bold", color="white", zorder=4)
    nxt = angles[(i+1) % n]
    x2, y2 = R*np.cos(nxt), R*np.sin(nxt)
    ax.add_patch(FancyArrowPatch((x, y), (x2, y2), connectionstyle="arc3,rad=0.18",
                                  arrowstyle="-|>", mutation_scale=18, color="#555555",
                                  linewidth=1.6, shrinkA=22, shrinkB=22, zorder=2))

ax.add_patch(Circle((0, 0), 1.65, facecolor="#0E4B5A", edgecolor="white", linewidth=3, zorder=3))
ax.text(0, 0.18, "Intelligent Maternal\nHealth Risk", ha="center", va="center",
        fontsize=11.5, fontweight="bold", color="white", zorder=4)
ax.text(0, -0.55, "Prediction & EWS", ha="center", va="center", fontsize=11.5, fontweight="bold", color="white", zorder=4)

ax.add_patch(FancyArrowPatch((R*np.cos(angles[-1])*0.6, R*np.sin(angles[-1])*0.6),
                              (R*np.cos(angles[0])*0.6, R*np.sin(angles[0])*0.6),
                              connectionstyle="arc3,rad=-0.7", arrowstyle="-|>", mutation_scale=16,
                              color="#E4572E", linewidth=2, linestyle="--", zorder=2))
ax.text(0, -3.1, "feedback: retrain on new clinic data", ha="center", fontsize=9.5,
        color="#E4572E", style="italic")

ax.set_xlim(-6, 6); ax.set_ylim(-6, 6); ax.axis("off")
ax.set_title("Circular Conceptual Model of the System", fontsize=15, fontweight="bold", pad=10)
plt.tight_layout(); plt.savefig("conceptual_model_circular.png", dpi=200); plt.show()

**1b. Tree Structure: Models → Hyperparameter Tuning → SHAP**

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

def box(x, y, w, h, text, color, fontsize=10, fontcolor="white"):
    ax.add_patch(FancyBboxPatch((x-w/2, y-h/2), w, h, boxstyle="round,pad=0.08,rounding_size=0.12",
                                 facecolor=color, edgecolor="white", linewidth=2, zorder=3))
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize, fontweight="bold", color=fontcolor, zorder=4)

def link(x1, y1, x2, y2, color="#888888"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=15,
                                  color=color, linewidth=1.6, shrinkA=8, shrinkB=8, zorder=2))

model_names = ["Logistic\nRegression", "K-Nearest\nNeighbors", "Random\nForest", "Gradient\nBoosting"]
model_colors = ["#4C78A8", "#72B7B2", "#54A24B", "#E45756"]
xs = [1.3, 4.1, 6.9, 9.7]

box(5.5, 7.2, 4.6, 0.9, "Scaled (+ SMOTE if needed)\nTraining Data", "#0E4B5A", 11)
for x, name, c in zip(xs, model_names, model_colors):
    link(5.5, 6.75, x, 5.7)
    box(x, 5.3, 2.3, 0.95, name, c, 10)

for x, c in zip(xs, model_colors):
    link(x, 4.85, x, 4.1)
    box(x, 3.7, 2.3, 0.85, "Tuning on\nValidation Set", "#8E7CC3", 8.7)
    link(x, 3.28, 5.5, 2.55)

box(5.5, 2.1, 4.8, 0.9, "Rank by Accuracy, F1, Sensitivity,\nSpecificity & ROC-AUC", "#F0A500", 10.3, "black")
link(5.5, 1.65, 5.5, 0.95)
box(5.5, 0.5, 3.4, 0.85, "Best Model Selected", "#2E9E6D", 11.5)

link(5.5, 0.08, 2.6, -0.95)
link(5.5, 0.08, 8.4, -0.95)
box(2.6, -1.35, 3.0, 0.85, "SHAP Global\nExplanation", "#9C6ADE", 9.8)
box(8.4, -1.35, 3.0, 0.85, "SHAP Local\nExplanation", "#9C6ADE", 9.8)
link(2.6, -1.78, 5.0, -2.75); link(8.4, -1.78, 6.0, -2.75)
box(5.5, -3.15, 5.4, 0.85, "Risk Level + Early Warning Alert", "#E4572E", 10.5)

ax.set_xlim(-0.3, 11.3); ax.set_ylim(-3.8, 8); ax.axis("off")
ax.set_title("Model Comparison → Hyperparameter Tuning → Explainability (Tree View)",
              fontsize=14.5, fontweight="bold", pad=10)
plt.tight_layout(); plt.savefig("tree_model_shap.png", dpi=200); plt.show()

## 2. Dataset

In [ ]:
m_df = pd.read_csv("Maternal_Health_Risk_RAW_UNCLEANED.csv", encoding="utf-8-sig")
print(f"Shape: {m_df.shape}  ->  {m_df.shape[1] - 1} features + the RiskLevel target")
m_df.head()

In [ ]:
m_df.info()
print("\nRiskLevel as typed in the raw file (note the inconsistent spellings):")
print(m_df["RiskLevel"].value_counts())
print("\nMissing values per column:")
print(m_df.isna().sum()[m_df.isna().sum() > 0])

In [ ]:
m_df.describe().T

## 2b. Data Cleaning

The raw file is deliberately messy, so it is cleaned **before** any analysis. Problems found and how each is handled:

| Problem | Fix |
|---|---|
| `RiskLevel` is typed 15 different ways (`LOW RISK`, ` low risk`, `Mid Risk`, ...) | strip spaces + lower-case, giving 3 classes |
| `SmokingAlcoholExposure` and `PreviousComplications` use 10 spellings (`Yes`, `Y`, `1`, `no`, `N`, `0`, ...) | map all to 0 / 1 |
| Exact duplicate rows | drop them |
| Impossible values (Age 0 or 115, HeartRate 300, BloodSugar 0 or 45, BMI 0 or 70, ...) | set to missing (NaN) using plausible clinical ranges |
| Missing values | **not** filled here; a median imputer inside the model pipeline fills them, learning the medians from the training data only (no leakage) |

In [ ]:
raw_rows = len(m_df)

# 1. RiskLevel labels
m_df["RiskLevel"] = m_df["RiskLevel"].str.strip().str.lower()

# 2. Yes / No columns -> 0 / 1
YES_NO = {"yes": 1, "y": 1, "1": 1, "no": 0, "n": 0, "0": 0}
BINARY = ["SmokingAlcoholExposure", "PreviousComplications"]
for c in BINARY:
    m_df[c] = m_df[c].astype(str).str.strip().str.lower().map(YES_NO)

# 3. exact duplicates
n_dup = int(m_df.duplicated().sum())
m_df = m_df.drop_duplicates().reset_index(drop=True)

# 4. impossible values -> NaN  (plausible clinical range for each measurement)
PLAUSIBLE = {"Age": (12, 60), "SystolicBP": (70, 200), "DiastolicBP": (40, 130),
             "BloodSugar": (3, 25), "BodyTemperature": (94, 106), "HeartRate": (40, 180),
             "BMI": (12, 60), "HemoglobinLevel": (5, 18), "GestationalAge": (4, 42),
             "WeightGain": (-5, 30), "PreviousPregnancies": (0, 15), "AntenatalVisitCount": (0, 20)}
set_to_nan = {}
for col, (lo, hi) in PLAUSIBLE.items():
    bad = m_df[col].notna() & ~m_df[col].between(lo, hi)
    set_to_nan[col] = int(bad.sum())
    m_df.loc[bad, col] = np.nan

CONT = [c for c in m_df.columns if c not in BINARY + ["RiskLevel"]]   # the 12 continuous / count features

print(f"Rows: {raw_rows} -> {len(m_df)}  ({n_dup} duplicates removed)")
print("Impossible values set to missing:", {k: v for k, v in set_to_nan.items() if v})
print("\nRiskLevel classes:", m_df["RiskLevel"].value_counts().to_dict())
print("Yes/No columns:", {c: m_df[c].value_counts().to_dict() for c in BINARY})
miss = m_df.isna().sum()
print(f"\nMissing values still to impute: {int(miss.sum())} cells in {int((miss > 0).sum())} columns")
display(miss[miss > 0].to_frame("Missing").assign(**{"Share (%)": lambda d: (d["Missing"] / len(m_df) * 100).round(1)}))

## 3. Exploratory Data Analysis

Distribution and outlier plots on the **cleaned** data, followed by Plotly-based views. All 14 features are shown.

**Feature distributions by risk level**

In [ ]:
NUM_FEATURES = [c for c in m_df.columns if c != "RiskLevel"]          # all 14 features
fig, axes = plt.subplots(nrows=5, ncols=3, figsize=(16, 20))
for ax, column in zip(axes.flatten(), NUM_FEATURES):
    sns.histplot(data=m_df, x=column, kde=column not in BINARY, hue="RiskLevel", hue_order=ORDER,
                 multiple="stack", palette=RISK_COLORS, element="bars", ax=ax)
    ax.set_title(column, fontsize=13)
for ax in axes.flatten()[len(NUM_FEATURES):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig("maternal_features_description.png", dpi=150); plt.show()

**Boxplots to check outliers (after cleaning)**

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(14, 14))
for ax, column in zip(axes.flatten(), CONT):
    sns.boxplot(y=m_df[column], color="#4682B4", ax=ax)
    ax.set_title(column, fontsize=13)
plt.tight_layout(); plt.savefig("maternal_outliers.png", dpi=150); plt.show()

**EDA extra 1 — Interactive correlation heatmap (Plotly)**

In [ ]:
numeric_df = m_df.drop(columns="RiskLevel")
corr = numeric_df.corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                 title="Feature Correlation Heatmap")
fig.update_layout(height=760)
fig.show()

**EDA extra 2 — Risk-level composition sunburst (Age band → Risk)**

In [ ]:
age_bins = pd.cut(m_df["Age"], bins=[9, 19, 29, 39, 49, 70],
                   labels=["10-19", "20-29", "30-39", "40-49", "50+"])
sb_df = m_df.assign(AgeBand=age_bins).dropna(subset=["AgeBand"])
fig = px.sunburst(sb_df, path=["AgeBand", "RiskLevel"], color="RiskLevel",
                   color_discrete_map=RISK_COLORS,
                   title="Risk Level Composition within Each Maternal Age Band")
fig.update_layout(height=560)
fig.show()

**EDA extra 3 — 3D scatter: Blood Sugar, Diastolic BP & Age by risk**

In [ ]:
fig = px.scatter_3d(m_df.dropna(subset=["BloodSugar", "DiastolicBP", "Age"]), x="BloodSugar", y="DiastolicBP", z="Age", color="RiskLevel",
                     color_discrete_map=RISK_COLORS, category_orders={"RiskLevel": ORDER},
                     opacity=0.75, size_max=6,
                     title="Blood Sugar, Diastolic BP & Age by Risk Level")
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=620)
fig.show()

**EDA extra 4 — Parallel coordinates across all clinical features**

In [ ]:
pc_df = m_df.dropna(subset=CONT).copy()
pc_df["RiskCode"] = pc_df["RiskLevel"].map({"low risk": 0, "mid risk": 1, "high risk": 2})
fig = px.parallel_coordinates(pc_df, dimensions=CONT,
                               color="RiskCode", color_continuous_scale=[COLORS[0], COLORS[1], COLORS[2]],
                               title="Parallel Coordinates of Clinical Features (0=Low, 1=Mid, 2=High risk)")
fig.update_layout(height=560)
fig.show()

**EDA extra 5 — Mean feature profile per risk level (radar chart)**

In [ ]:
feat_cols = CONT
means = m_df.groupby("RiskLevel")[feat_cols].mean().reindex(ORDER)
means_norm = (means - m_df[feat_cols].min()) / (m_df[feat_cols].max() - m_df[feat_cols].min())

fig = go.Figure()
for lvl in ORDER:
    vals = means_norm.loc[lvl].tolist()
    fig.add_trace(go.Scatterpolar(r=vals + vals[:1], theta=feat_cols + feat_cols[:1],
                                   fill="toself", name=lvl, line_color=RISK_COLORS[lvl]))
fig.update_layout(title="Normalised Mean Clinical Profile per Risk Level",
                   polar=dict(radialaxis=dict(visible=True, range=[0, 1])), height=520)
fig.show()

**EDA extra 6 — Blood sugar distribution by risk level (violin, Plotly)**

In [ ]:
fig = px.violin(m_df, x="RiskLevel", y="BloodSugar", color="RiskLevel", box=True, points="outliers",
                 category_orders={"RiskLevel": ORDER}, color_discrete_map=RISK_COLORS,
                 title="Blood Sugar Distribution by Risk Level")
fig.update_layout(height=480, showlegend=False)
fig.show()

## 4. Preprocessing & Data Splitting

All **14 features** are kept. No feature is dropped for collinearity: the highest correlation between any two features is printed below and is low (the old dataset's SystolicBP / DiastolicBP problem does not appear here).

Missing values are filled with the **median** by an imputer *inside* the model pipeline (Section 6), so the medians come from the training data only.

Because **no cross-validation** is used, the data is split three ways (stratified, so every split keeps the same class mix):

| Split | Share | Used for |
|---|---|---|
| Train | 60 % | fitting models while tuning |
| Validation | 20 % | comparing hyperparameters (replaces CV) |
| Test | 20 % | the final, untouched evaluation |

In [ ]:
m_df["RiskLevel"] = m_df["RiskLevel"].map({"low risk": 0, "mid risk": 1, "high risk": 2})

FEATURES = [c for c in m_df.columns if c != "RiskLevel"]
X, y = m_df[FEATURES], m_df["RiskLevel"]

pair_corr = X.corr().abs().where(~np.eye(len(FEATURES), dtype=bool))
print(f"Features: {len(FEATURES)}")
print(f"Highest absolute correlation between two features: {pair_corr.max().max():.2f}")
print(f"Missing cells to be imputed: {int(X.isna().sum().sum())}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

In [ ]:
# carve a validation set out of the training data  ->  60 / 20 / 20
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                            random_state=RANDOM_STATE, stratify=y_train)
print(f"Train: {X_tr.shape}   Validation: {X_val.shape}   Test: {X_test.shape}")

## 5. Class Imbalance Check

SMOTE is only worth using if the imbalance actually hurts. Here we measure it first, and later let the **validation set decide** (each model is tuned with and without SMOTE). SMOTE is always applied to the *training portion only*, never to validation/test data.

In [ ]:
counts = y_train.map(dict(enumerate(LABELS))).value_counts().reindex(LABELS)
imbalance = pd.DataFrame({"Count": counts, "Share (%)": (counts / counts.sum() * 100).round(1)})
display(imbalance)
ratio = counts.max() / counts.min()
print(f"Imbalance ratio (largest : smallest class) = {ratio:.2f} : 1")
print("-> mild imbalance: SMOTE is treated as a tuning option rather than assumed." if ratio < 3
      else "-> strong imbalance: SMOTE / class weighting is strongly recommended.")

ax = counts.plot(kind="bar", color=COLORS, figsize=(5, 3.2), rot=0)
ax.set_title("Training-set class distribution"); ax.set_ylabel("Patients")
plt.tight_layout(); plt.show()

## 6. Helper Functions & Baseline Models

`make_pipe` builds *Median imputer → Scaler → (optional SMOTE) → Classifier*. Imputer, scaler and SMOTE are fitted on the training data only (no leakage).

`score_model` returns every metric used in the slides plus **sensitivity** and **specificity**. For a 3-class problem these are computed one-vs-rest from the confusion matrix for each class and macro-averaged:

* Sensitivity = TP / (TP + FN)
* Specificity = TN / (TN + FP)
* ROC-AUC = one-vs-rest, macro-averaged
* Precision / Recall / F1 = weighted average (so weighted Recall equals Accuracy)

In [ ]:
def make_pipe(clf, use_smote=False):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("clf", clf))
    return Pipeline(steps)


def score_model(pipe, X, y):
    """All metrics for one fitted pipeline on one dataset."""
    yp, pr = pipe.predict(X), pipe.predict_proba(X)
    cm = confusion_matrix(y, yp, labels=[0, 1, 2])
    tp = np.diag(cm); fn = cm.sum(axis=1) - tp; fp = cm.sum(axis=0) - tp; tn = cm.sum() - tp - fn - fp
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    p, r, f, _ = precision_recall_fscore_support(y, yp, average="weighted", zero_division=0)
    metrics = {"Accuracy": accuracy_score(y, yp), "Precision": p, "Recall": r, "F1": f,
               "Sensitivity": sens.mean(), "Specificity": spec.mean(),
               "ROC-AUC": roc_auc_score(y, pr, multi_class="ovr", average="macro"),
               "Macro F1": f1_score(y, yp, average="macro"),
               "High-risk Sensitivity": sens[2], "High-risk Specificity": spec[2]}
    return {"metrics": metrics, "cm": cm, "pred": yp, "proba": pr, "sens": sens, "spec": spec}


models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

**6a. Baseline run** — default hyperparameters, no SMOTE, trained on the 60 % train split and scored on the validation split.

In [ ]:
SHOW = ["Accuracy", "Precision", "Recall", "F1", "Sensitivity", "Specificity", "ROC-AUC"]

baseline = {}
for name, clf in models.items():
    pipe = make_pipe(clone(clf), use_smote=False).fit(X_tr, y_tr)
    baseline[name] = score_model(pipe, X_val, y_val)["metrics"]

baseline_df = pd.DataFrame(baseline).T
print("Baseline models - validation set")
display(baseline_df[SHOW + ["Macro F1"]].round(3))

## 7. Hyperparameter Tuning (hold-out validation, no cross-validation)

Every combination in each model's grid is trained on the **train split** and scored once on the **validation split**. SMOTE on/off is part of the search, so imbalance handling is only used where it helps. The winner per model is chosen by **validation macro-F1** (accuracy breaks ties) - macro-F1 weighs the smaller classes equally, which suits an imbalanced problem. The test set is not touched here.

In [ ]:
grids = {
    "Logistic Regression": {"C": [0.01, 0.1, 1, 10, 100]},
    "K-Nearest Neighbors": {"n_neighbors": [3, 5, 9, 15], "weights": ["uniform", "distance"]},
    "Random Forest": {"n_estimators": [200, 400], "max_depth": [None, 10, 20], "min_samples_leaf": [1, 3]},
    "Gradient Boosting": {"n_estimators": [150], "max_depth": [3, 5], "learning_rate": [0.05, 0.1]},
}

trials = []
for name, clf in models.items():
    for use_smote in (False, True):
        for params in ParameterGrid(grids[name]):
            pipe = make_pipe(clone(clf).set_params(**params), use_smote).fit(X_tr, y_tr)
            m = score_model(pipe, X_val, y_val)["metrics"]
            trials.append({"Model": name, "SMOTE": use_smote, "Params": params,
                           "Val Accuracy": m["Accuracy"], "Val Macro F1": m["Macro F1"]})
    print(f"tuned {name}")
trials = pd.DataFrame(trials)
print(f"{len(trials)} configurations evaluated")

In [ ]:
best_cfg, summary = {}, []
for name in models:
    top = (trials[trials.Model == name]
           .sort_values(["Val Macro F1", "Val Accuracy"], ascending=False).iloc[0])
    best_cfg[name] = {"Params": top["Params"], "SMOTE": bool(top["SMOTE"])}
    summary.append({"Model": name, "Best hyperparameters": top["Params"], "SMOTE chosen": bool(top["SMOTE"]),
                    "Baseline Val Macro F1": baseline[name]["Macro F1"],
                    "Tuned Val Macro F1": top["Val Macro F1"], "Tuned Val Accuracy": top["Val Accuracy"]})
tuning_summary = pd.DataFrame(summary).set_index("Model")
print("Tuning summary")
display(tuning_summary.round(3))

print("Does SMOTE help? Best validation macro-F1 per model, without vs with SMOTE")
smote_effect = trials.pivot_table(index="Model", columns="SMOTE", values="Val Macro F1", aggfunc="max")
smote_effect.columns = ["No SMOTE", "With SMOTE"]
smote_effect["Difference"] = smote_effect["With SMOTE"] - smote_effect["No SMOTE"]
display(smote_effect.round(3))

## 8. Final Training & Test-Set Evaluation

Each model is retrained with its best hyperparameters (and SMOTE setting) on **train + validation (80 %)**, then evaluated once on the untouched **test set (20 %)**.

In [ ]:
final, evals, rows = {}, {}, []
for name, cfg in best_cfg.items():
    pipe = make_pipe(clone(models[name]).set_params(**cfg["Params"]), cfg["SMOTE"]).fit(X_train, y_train)
    final[name] = pipe
    evals[name] = score_model(pipe, X_test, y_test)
    rows.append({"Model": name, "SMOTE": cfg["SMOTE"], "Train Accuracy": pipe.score(X_train, y_train),
                 **evals[name]["metrics"]})

results = pd.DataFrame(rows).set_index("Model")
results["Train-Test Gap"] = results["Train Accuracy"] - results["Accuracy"]
print("TEST-SET RESULTS")
display(results[["SMOTE", "Train Accuracy"] + SHOW + ["High-risk Sensitivity", "Train-Test Gap"]].round(3))

**8a. Per-class sensitivity and specificity (one-vs-rest)**

In [ ]:
per_class = pd.concat({name: pd.DataFrame({"Sensitivity": e["sens"], "Specificity": e["spec"]}, index=LABELS)
                       for name, e in evals.items()}, names=["Model", "Class"])
display(per_class.round(3))

**8b. Classification reports**

In [ ]:
for name, e in evals.items():
    print(f"===== {name} =====")
    print(classification_report(y_test, e["pred"], target_names=LABELS, zero_division=0))

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, e) in zip(axes.flatten(), evals.items()):
    sns.heatmap(e["cm"], annot=True, fmt="d", cmap="GnBu", cbar=False, ax=ax,
                xticklabels=LABELS, yticklabels=LABELS)
    ax.set_title(f"{name}  (acc={e['metrics']['Accuracy']:.3f})", fontsize=11)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

In [ ]:
fig = go.Figure()
for metric in SHOW:
    fig.add_bar(x=results.index, y=results[metric], name=metric)
fig.update_layout(barmode="group", title="Model Comparison on the Held-Out Test Set",
                   yaxis_title="Score", yaxis_range=[0.4, 1.0], height=480)
fig.show()

## 10. Best Model Selection

Models are ranked on test Accuracy, F1, Sensitivity, Specificity, ROC-AUC and **High-risk sensitivity** (missing a High-risk pregnancy is the costliest error). The model with the lowest mean rank is selected.

In [ ]:
rank_cols = ["Accuracy", "F1", "Sensitivity", "Specificity", "ROC-AUC", "High-risk Sensitivity"]
ranks = results[rank_cols].rank(ascending=False, method="min")
ranks["Mean Rank"] = ranks.mean(axis=1)
ranks = ranks.sort_values("Mean Rank")
display(ranks)

best_name = ranks.index[0]
best_pipe = final[best_name]
print(f"\nSelected model: {best_name}")
print("Hyperparameters:", best_cfg[best_name]["Params"], "| SMOTE:", best_cfg[best_name]["SMOTE"])

In [ ]:
# ROC curves (one-vs-rest) for the selected model
y_bin = label_binarize(y_test, classes=[0, 1, 2])
proba = evals[best_name]["proba"]
fig, ax = plt.subplots(figsize=(6, 5))
for c, (lab, col) in enumerate(zip(LABELS, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, c], proba[:, c])
    ax.plot(fpr, tpr, color=col, lw=2, label=f"{lab} (AUC = {roc_auc_score(y_bin[:, c], proba[:, c]):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("1 - Specificity (False Positive Rate)"); ax.set_ylabel("Sensitivity (True Positive Rate)")
ax.set_title(f"ROC curves: {best_name}"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("roc_best_model.png", dpi=150); plt.show()

## 11. SHAP Explainability (applied to the best model)

SHAP is the first of the three XAI techniques compared in Section 12. The cells below build the SHAP explanations for the whole test set and plot them.

In [ ]:
prep = SkPipeline([("imputer", best_pipe.named_steps["imputer"]), ("scaler", best_pipe.named_steps["scaler"])])
clf = best_pipe.named_steps["clf"]
X_bg = pd.DataFrame(prep.transform(X_train), columns=FEATURES)
X_te = pd.DataFrame(prep.transform(X_test), columns=FEATURES, index=X_test.index)

explain = None
if isinstance(clf, (RandomForestClassifier, GradientBoostingClassifier)):
    try:        # fast, exact explainer for tree models (not supported for multi-class Gradient Boosting in the newest SHAP)
        explainer = shap.TreeExplainer(clf)
        explain = lambda data: explainer(data, check_additivity=False)
        explain(X_bg.iloc[:3])
    except Exception:
        explain = None
if explain is None:   # Logistic Regression / KNN / fallback: model-agnostic explainer
    explainer = shap.Explainer(clf.predict_proba, X_bg.sample(100, random_state=RANDOM_STATE))
    explain = lambda data: explainer(data)

shap_exp = explain(X_te)
shap_values = shap_exp.values
base_values = np.broadcast_to(shap_exp.base_values, (len(X_te), shap_values.shape[2]))
print("SHAP values shape (samples, features, classes):", shap_values.shape)

**11a. SHAP importance table**

In [ ]:
imp = np.abs(shap_values).mean(axis=0)                       # (features, classes)
shap_table = pd.DataFrame(imp, index=FEATURES, columns=[f"{l} risk" for l in LABELS])
shap_table["Total"] = shap_table.sum(axis=1)
shap_table = shap_table.sort_values("Total", ascending=False)
shap_table["Share (%)"] = shap_table["Total"] / shap_table["Total"].sum() * 100
display(shap_table.round(4))

In [ ]:
imp = np.abs(shap_values).mean(axis=0)
order = np.argsort(imp.sum(axis=1))
fig, ax = plt.subplots(figsize=(8, 4.5))
left = np.zeros(len(FEATURES))
for c, (lab, col) in enumerate(zip(LABELS, COLORS)):
    ax.barh(np.array(FEATURES)[order], imp[order, c], left=left, color=col, label=f"{lab} risk", edgecolor="white")
    left += imp[order, c]
ax.set_xlabel("Mean |SHAP value|"); ax.set_title(f"Global Feature Importance: {best_name}")
ax.legend(); plt.tight_layout(); plt.savefig("shap_global_importance.png", dpi=200); plt.show()

In [ ]:
for c, lab in enumerate(LABELS):
    shap.summary_plot(shap_values[:, :, c], X_test, feature_names=FEATURES, show=False)
    plt.title(f"SHAP summary: effect of each feature on {lab}-risk prediction")
    plt.tight_layout(); plt.savefig(f"shap_summary_{lab.lower()}.png", dpi=150); plt.show()

**11b. Local explanations (one patient per predicted class)**

In [ ]:
y_pred_best = best_pipe.predict(X_test)
for c, lab in enumerate(LABELS):
    idx = np.where(y_pred_best == c)[0][0]
    prob = best_pipe.predict_proba(X_test.iloc[[idx]])[0]
    exp = shap.Explanation(values=shap_values[idx, :, c], base_values=base_values[idx, c],
                            data=X_test.iloc[idx].values, feature_names=FEATURES)
    shap.plots.waterfall(exp, show=False)
    plt.title(f"Patient #{idx}: predicted {lab} risk (p={prob[c]:.2f}, true={LABELS[y_test.iloc[idx]]})")
    plt.tight_layout(); plt.savefig(f"shap_local_{lab.lower()}.png", dpi=150, bbox_inches="tight"); plt.show()

## 12. Comparing Three XAI Techniques (SHAP vs LIME vs Permutation Importance)

The selected model (Gradient Boosting) is a black box, so three explainability techniques are compared. The best one is then used inside the predictive system (Section 13).

| Technique | Scope | Idea |
|---|---|---|
| **SHAP** | local + global | Shapley values from game theory: each feature receives a fair share of the gap between this patient's prediction and the average prediction. The shares add up exactly to that gap. |
| **LIME** | local | Builds many slightly changed copies of one patient, asks the model about each copy, then fits a small linear model around that patient. The linear model's weights are the explanation. |
| **Permutation Feature Importance (PFI)** | global | Shuffles one feature at a time on the test set and measures how much the model's macro-F1 drops. It says which features matter *overall*, not why *this* patient got her result. |

**How "best performing" is measured** (same 150 test patients for every technique, explaining the class the model predicted):

1. **Faithfulness (deletion test).** Features are switched off one by one (replaced by the training average) in the order the technique ranks them. If the explanation is faithful, the probability of the predicted class collapses quickly. Reported as the area under that curve (*lower is better*) and as the probability lost after removing the top 3 features (*higher is better*).
2. **Stability.** Each technique is run twice with different random seeds. A trustworthy explanation names the same top-3 features both times.
3. **Speed.** Seconds needed per explanation.

**Selection score** = 50 % faithfulness (two metrics) + 30 % stability + 20 % speed. Each metric is ranked across the three techniques (best = 3 points). A random feature order is drawn only as a sanity-check baseline: a real technique must beat it. The predictive system explains *one patient at a time*, so the technique used there must also be able to explain a single patient.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
import time

N_EVAL = 150                                             # patients used for the comparison
rng = np.random.RandomState(RANDOM_STATE)
eval_idx = np.sort(rng.choice(len(X_te), N_EVAL, replace=False))
Xe = X_te.iloc[eval_idx]                                 # scaled + imputed test patients
proba_e = clf.predict_proba(Xe.values)
pred_e = proba_e.argmax(axis=1)                          # class the model predicted for each patient
rows = np.arange(N_EVAL)
K = len(FEATURES)
is_tree_shap = isinstance(explainer, shap.TreeExplainer)

# ---- the three techniques: each returns an (N_EVAL x features) matrix; higher = more important for the predicted class
def attr_shap(seed):
    if is_tree_shap:                                      # exact and deterministic
        ex = explain
    else:                                                 # sampling-based fallback: seed changes the samples
        bg = X_bg.sample(100, random_state=seed)
        try:
            e = shap.Explainer(clf.predict_proba, bg, seed=seed)
        except TypeError:
            np.random.seed(seed); e = shap.Explainer(clf.predict_proba, bg)
        ex = lambda data: e(data)
    sv = ex(Xe).values                                    # (patients, features, classes)
    return sv[rows, :, pred_e]

def attr_lime(seed):
    lime_ex = LimeTabularExplainer(X_bg.values, feature_names=FEATURES, class_names=LABELS,
                                   mode="classification", random_state=seed)
    A = np.zeros((N_EVAL, K))
    for i, (row, c) in enumerate(zip(Xe.values, pred_e)):
        e = lime_ex.explain_instance(row, clf.predict_proba, labels=(int(c),), num_features=K, num_samples=1000)
        for j, w in e.as_map()[int(c)]:
            A[i, j] = w
    return A

def attr_pfi(seed):
    r = permutation_importance(best_pipe, X_test, y_test, scoring="f1_macro", n_repeats=10, random_state=seed)
    return np.tile(r.importances_mean, (N_EVAL, 1))       # global: same ranking for every patient

TECHNIQUES = {"SHAP": attr_shap, "LIME": attr_lime, "Permutation Importance": attr_pfi}

# ---- metrics
def deletion_curve(A):
    """Probability of the predicted class as the top-k ranked features are switched off, k = 0..K."""
    cur = Xe.values.copy()
    P = np.zeros((N_EVAL, K + 1)); P[:, 0] = proba_e[rows, pred_e]
    order = np.argsort(-A, axis=1)
    for k in range(K):
        cur[rows, order[:, k]] = 0.0                      # 0 = training average (data are scaled)
        P[:, k + 1] = clf.predict_proba(cur)[rows, pred_e]
    return P

def curve_auc(P):  return float(((P[:, :-1] + P[:, 1:]) / 2).mean())
def top3_overlap(A, B):
    ta, tb = np.argsort(-A, axis=1)[:, :3], np.argsort(-B, axis=1)[:, :3]
    return float(np.mean([len(set(a) & set(b)) / 3 for a, b in zip(ta, tb)]))
def rank_corr(A, B):
    return float(np.nanmean([spearmanr(a, b)[0] for a, b in zip(A, B)]))

# ---- run every technique twice (two seeds); time the second run
runs, curves, rows_out = {}, {}, []
for name, fn in TECHNIQUES.items():
    A1 = fn(RANDOM_STATE)
    t0 = time.time(); A2 = fn(RANDOM_STATE + 1); secs = (time.time() - t0) / N_EVAL
    runs[name] = A1
    P = deletion_curve(A1); curves[name] = P
    rows_out.append({"Technique": name, "Faithfulness AUC (lower=better)": curve_auc(P),
                     "Prob. drop after top-3 removed (higher=better)": float(P[:, 0].mean() - P[:, 3].mean()),
                     "Top-3 stability (higher=better)": top3_overlap(A1, A2),
                     "Rank correlation between runs": rank_corr(A1, A2),
                     "Seconds per explanation (lower=better)": secs,
                     "Explains one patient": name != "Permutation Importance"})
    print(f"done: {name}")

rand_P = deletion_curve(np.random.RandomState(0).rand(N_EVAL, K)); curves["Random order (baseline)"] = rand_P
xai_table = pd.DataFrame(rows_out).set_index("Technique")
display(xai_table.round(4))
print(f"Random-order baseline: faithfulness AUC = {curve_auc(rand_P):.4f}, "
      f"probability drop after top-3 = {rand_P[:, 0].mean() - rand_P[:, 3].mean():.4f}")

In [ ]:
# ---- selection score (best on a metric = 3 points, worst = 1)
t = xai_table
n = len(t)
pts = pd.DataFrame({
    "Faithfulness AUC":        (n + 1) - t["Faithfulness AUC (lower=better)"].rank(ascending=True),
    "Prob. drop (top-3)":      (n + 1) - t["Prob. drop after top-3 removed (higher=better)"].rank(ascending=False),
    "Stability":               (n + 1) - t["Top-3 stability (higher=better)"].rank(ascending=False, method="min"),
    "Speed":                   (n + 1) - t["Seconds per explanation (lower=better)"].rank(ascending=True),
})
xai_score = (0.25 * pts["Faithfulness AUC"] + 0.25 * pts["Prob. drop (top-3)"]
             + 0.30 * pts["Stability"] + 0.20 * pts["Speed"]).rename("Weighted score (max 3)")
xai_rank = pd.concat([pts, xai_score], axis=1).sort_values("Weighted score (max 3)", ascending=False)
display(xai_rank.round(2))

best_overall = xai_rank.index[0]
local_ok = xai_rank.index[t.loc[xai_rank.index, "Explains one patient"].values]
best_xai = best_overall if best_overall in local_ok else local_ok[0]
print(f"\nBest-scoring technique overall : {best_overall}")
if best_xai != best_overall:
    print(f"It cannot explain a single patient, so the predictive system uses: {best_xai}")
print(f"Technique used in the predictive system: {best_xai}")

# ---- figures
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
palette = {"SHAP": "#1f77b4", "LIME": "#ff7f0e", "Permutation Importance": "#2ca02c", "Random order (baseline)": "#999999"}
for name, P in curves.items():
    axes[0].plot(range(K + 1), P.mean(axis=0), marker="o", ms=3, lw=2, label=name, color=palette[name],
                 ls="--" if "Random" in name else "-")
axes[0].set_xlabel("Number of top-ranked features switched off"); axes[0].set_ylabel("Mean probability of the predicted class")
axes[0].set_title("Deletion test: a faster fall = a more faithful explanation"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].barh(xai_rank.index[::-1], xai_rank["Weighted score (max 3)"][::-1],
             color=[palette[i] for i in xai_rank.index[::-1]])
axes[1].set_xlim(0, 3); axes[1].set_xlabel("Weighted score (max 3)"); axes[1].set_title("XAI technique ranking")
plt.tight_layout(); plt.savefig("xai_comparison.png", dpi=150); plt.show()

# ---- do the techniques agree on which features matter overall?
glob = pd.DataFrame({"SHAP": np.abs(runs["SHAP"]).mean(axis=0), "LIME": np.abs(runs["LIME"]).mean(axis=0),
                     "Permutation Importance": runs["Permutation Importance"][0]}, index=FEATURES)
print("Top-5 features by technique (global view):")
display(pd.DataFrame({c: glob[c].sort_values(ascending=False).index[:5] for c in glob}, index=range(1, 6)))
print("Rank agreement between techniques (Spearman):")
display(glob.corr(method="spearman").round(2))

In [ ]:
# ---- why the winner was chosen (numbers come from this run)
w = best_xai
others = [x for x in xai_table.index if x != w]
f_auc, f_drop = xai_table["Faithfulness AUC (lower=better)"], xai_table["Prob. drop after top-3 removed (higher=better)"]
stab, spd = xai_table["Top-3 stability (higher=better)"], xai_table["Seconds per explanation (lower=better)"]
print(f"REASONS FOR CHOOSING {w.upper()}\n" + "-" * 60)
print(f"1. Faithfulness: removing its top-3 features cut the predicted-class probability by {f_drop[w]:.3f} "
      f"(" + ", ".join(f"{o} {f_drop[o]:.3f}" for o in others) + f"; random {rand_P[:, 0].mean() - rand_P[:, 3].mean():.3f}).")
print(f"   Its deletion-curve AUC is {f_auc[w]:.3f} (" + ", ".join(f"{o} {f_auc[o]:.3f}" for o in others) + ").")
print(f"2. Stability: the same top-3 features were returned on {stab[w] * 100:.0f}% of patients across two random seeds "
      f"(" + ", ".join(f"{o} {stab[o] * 100:.0f}%" for o in others) + ").")
spd_rank = int(spd.rank(method="min")[w])
print(f"3. Speed: {spd[w]:.3f} s per explanation (" + ", ".join(f"{o} {spd[o]:.3f} s" for o in others) + f"); "
      f"speed rank {spd_rank} of {len(spd)}, " + ("fast enough" if spd[w] < 1 else "slow but acceptable")
      + " for explaining one patient at a time.")
agree = glob.corr(method="spearman").values[np.triu_indices(len(glob.columns), 1)].min()
print(f"4. Agreement: the three techniques rank the features similarly overall (lowest Spearman = {agree:.2f}), "
      f"so the choice rests on the patient-level measurements above rather than on a disagreement about what matters.")
print(f"5. Patient-level: it explains one patient at a time and shows the direction (raises / lowers risk), "
      f"which the predictive system needs to name the abnormal features behind each result.")

**Reading the comparison**

* **SHAP** is built on Shapley values, so its contributions always add up to the model's output and the same feature gets the same credit however the model is built. It gives both a per-patient and a whole-model view from one method.
* **LIME** explains one patient by fitting a small linear model around her, using random perturbations. The random sampling is why its answers can change slightly between runs, and its linear surrogate may not match the model's real behaviour near that patient.
* **Permutation Importance** is quick and easy to read, but it is global: it gives every patient the same ranking and cannot show that a high blood sugar raised *this* patient's risk.

The measured results above decide the choice; the technique with the best weighted score is the one the predictive system uses.

## 13. Predictive System (Risk Prediction + Early Warning Alert + Explanation)

Enter a patient's 14 measurements (Yes/No items as 1/0). The system returns:

1. the predicted risk level and class probabilities,
2. the early-warning alert (High risk, or Mid risk with a High-risk probability at or above the threshold),
3. the **abnormal features that led to the result**: measurements outside the normal clinical range *and* pushing the prediction toward the predicted class, ranked by the contribution computed with the best XAI technique from Section 12,
4. other abnormal values that were present but did not drive the result.

**Normal clinical ranges.** The ranges below are screening references taken from ACOG / ADA / IADPSG (blood pressure, glucose), CDC / ACOG (anaemia), WHO (antenatal visits), IOM (weight gain) and standard adult references (temperature, heart rate, BMI, age). They support decision-making and do **not** replace a clinician's judgement. Two assumptions: blood sugar is treated as a **non-fasting** reading (the dataset does not say), and weight gain is judged against the IOM expected gain for the patient's BMI group and gestational week.

In [ ]:
# ---------------- normal clinical ranges ----------------
FIXED_RANGES = {   # feature: (lowest normal, highest normal, text)
    "Age":                 (18, 34,     "18-34 years"),
    "SystolicBP":          (90, 139,    "90-139 mmHg"),
    "DiastolicBP":         (60, 89,     "60-89 mmHg"),
    "BloodSugar":          (3.9, 7.8,   "3.9-7.8 mmol/L (non-fasting reading)"),
    "BodyTemperature":     (97.0, 99.0, "97.0-99.0 deg F"),
    "HeartRate":           (60, 100,    "60-100 bpm at rest"),
    "BMI":                 (18.5, 24.99, "18.5-24.9 kg/m2"),
    "PreviousPregnancies": (0, 4,       "0-4 previous pregnancies"),
}
ABOVE_NOTES = {   # checked from the largest threshold down
    "Age": [(35, "advanced maternal age")],
    "SystolicBP": [(160, "severe-range hypertension"), (140, "hypertension range")],
    "DiastolicBP": [(110, "severe-range hypertension"), (90, "hypertension range")],
    "BloodSugar": [(11.1, "diabetes range"), (10.0, "reaches the 1-hour gestational-diabetes cut-off"), (7.8, "raised blood sugar")],
    "BodyTemperature": [(100.4, "fever"), (99.0, "slightly raised temperature")],
    "HeartRate": [(100, "tachycardia")],
    "BMI": [(30, "obese"), (25, "overweight")],
    "PreviousPregnancies": [(5, "grand multiparity")],
}
BELOW_NOTES = {"Age": "adolescent pregnancy", "SystolicBP": "low blood pressure", "DiastolicBP": "low blood pressure",
               "BloodSugar": "low blood sugar", "BodyTemperature": "below normal temperature",
               "HeartRate": "slow heart rate (bradycardia)", "BMI": "underweight"}
BINARY_NOTES = {"SmokingAlcoholExposure": "smoking / alcohol exposure present",
                "PreviousComplications": "previous pregnancy complications"}
WHO_CONTACT_WEEKS = (12, 20, 26, 30, 34, 36, 38, 40)             # WHO 2016: 8 antenatal contacts
IOM_TOTAL = {"underweight": (12.5, 18), "normal weight": (11.5, 16),      # kg over the whole pregnancy (IOM 2009)
             "overweight": (7, 11.5), "obese": (5, 9)}
IOM_RATE = {"underweight": (0.44, 0.58), "normal weight": (0.35, 0.50),   # kg / week in trimesters 2-3 (IOM 2009)
            "overweight": (0.23, 0.33), "obese": (0.17, 0.27)}


def bmi_group(bmi):
    return "underweight" if bmi < 18.5 else "normal weight" if bmi < 25 else "overweight" if bmi < 30 else "obese"


def expected_weight_gain(bmi, ga):
    """IOM: 0.5-2.0 kg in the first 13 weeks, then a BMI-specific weekly rate."""
    lo_rate, hi_rate = IOM_RATE[bmi_group(bmi)]
    first, extra = min(ga, 13) / 13, max(ga - 13, 0)
    return 0.5 * first + extra * lo_rate, 2.0 * first + extra * hi_rate


def clinical_status(feature, value, patient):
    """Return (status, normal-range text, note). status = 'below normal' | 'normal' | 'above normal'."""
    v = float(value)
    if feature in FIXED_RANGES:
        lo, hi, txt = FIXED_RANGES[feature]
        if v < lo:
            return "below normal", txt, BELOW_NOTES.get(feature, "")
        if v > hi:
            return "above normal", txt, next((n for t, n in ABOVE_NOTES.get(feature, []) if v >= t), "")
        return "normal", txt, ""
    ga = float(patient["GestationalAge"])
    if feature == "HemoglobinLevel":
        lo = 10.5 if 14 <= ga < 28 else 11.0                    # 2nd trimester cut-off is 10.5, otherwise 11.0
        txt = f"{lo:.1f} g/dL or more (anaemia below this)"
        if v < lo:
            return "below normal", txt, "severe anaemia" if v < 7 else "moderate anaemia" if v < 10 else "mild anaemia"
        return "normal", txt, ""
    if feature == "GestationalAge":
        txt = "up to 41 weeks (term = 37-41 weeks)"
        return ("above normal", txt, "post-term pregnancy") if v >= 42 else ("normal", txt, "")
    if feature == "WeightGain":
        grp = bmi_group(float(patient["BMI"]))
        lo, _ = expected_weight_gain(float(patient["BMI"]), ga)         # minimum expected by this week
        hi = IOM_TOTAL[grp][1]                                           # maximum for the whole pregnancy
        txt = f"at least {lo:.1f} kg by week {ga:g} and no more than {hi:g} kg in total, for a {grp} BMI (IOM)"
        if v < lo:
            return "below normal", txt, "less weight gain than recommended"
        if v > hi:
            return "above normal", txt, "more weight gain than recommended"
        return "normal", txt, ""
    if feature == "AntenatalVisitCount":
        need = sum(ga >= w for w in WHO_CONTACT_WEEKS)
        txt = f"at least {need} visit(s) by week {ga:g} (WHO: 8 contacts at weeks 12, 20, 26, 30, 34, 36, 38, 40)"
        return ("below normal", txt, "fewer antenatal visits than recommended") if v < need else ("normal", txt, "")
    if feature in BINARY_NOTES:
        return ("above normal", "No (0)", BINARY_NOTES[feature]) if v >= 1 else ("normal", "No (0)", "")
    raise KeyError(feature)

In [ ]:
ALERT_THRESHOLD = 0.30
UNITS = {"Age": "years", "SystolicBP": "mmHg", "DiastolicBP": "mmHg", "BloodSugar": "mmol/L",
         "BodyTemperature": "deg F", "HeartRate": "bpm", "BMI": "kg/m2", "HemoglobinLevel": "g/dL",
         "GestationalAge": "weeks", "SmokingAlcoholExposure": "1 = Yes, 0 = No",
         "PreviousComplications": "1 = Yes, 0 = No", "WeightGain": "kg",
         "PreviousPregnancies": "count", "AntenatalVisitCount": "visits"}
TRAIN_RANGE = X.agg(["min", "max"])
lime_single = LimeTabularExplainer(X_bg.values, feature_names=FEATURES, class_names=LABELS,
                                   mode="classification", random_state=RANDOM_STATE)


def local_contributions(x, pred):
    """How much each feature pushed this patient toward class `pred`, using the best XAI technique (Section 12)."""
    z = pd.DataFrame(prep.transform(x), columns=FEATURES)
    if best_xai == "LIME":
        e = lime_single.explain_instance(z.values[0], clf.predict_proba, labels=(pred,), num_features=len(FEATURES), num_samples=1000)
        w = np.zeros(len(FEATURES))
        for j, wt in e.as_map()[pred]:
            w[j] = wt
        return w
    return explain(z).values[0, :, pred]                      # SHAP


def assess_patient(patient, threshold=ALERT_THRESHOLD):
    missing = [f for f in FEATURES if f not in patient]
    if missing:
        raise ValueError(f"Missing input(s): {missing}")
    x = pd.DataFrame([patient])[FEATURES].astype(float)
    notes = [f"{f}={x.at[0, f]:g} is outside the training range "
             f"({TRAIN_RANGE.at['min', f]:g}-{TRAIN_RANGE.at['max', f]:g})"
             for f in FEATURES if not TRAIN_RANGE.at["min", f] <= x.at[0, f] <= TRAIN_RANGE.at["max", f]]

    prob = best_pipe.predict_proba(x)[0]
    pred = int(prob.argmax())
    alert = (pred == 2) or (pred == 1 and prob[2] >= threshold)

    contrib = local_contributions(x, pred)
    push = max(contrib[contrib > 0].sum(), 1e-12)               # total push toward the predicted class
    findings = []
    for f, c in zip(FEATURES, contrib):
        status, normal_txt, note = clinical_status(f, x.at[0, f], patient)
        findings.append({"feature": f, "value": float(x.at[0, f]), "unit": UNITS[f], "status": status,
                         "normal": normal_txt, "note": note, "contribution": float(c),
                         "share": float(max(c, 0) / push * 100)})
    by_push = sorted(findings, key=lambda d: -d["contribution"])
    drivers = [d for d in by_push if d["status"] != "normal" and d["share"] >= 5]        # abnormal AND pushing toward the result
    others = [d for d in by_push if d["status"] != "normal" and d["share"] < 5]          # abnormal, but not a driver
    normal_help = [d for d in by_push if d["status"] == "normal" and d["share"] >= 5][:3]
    return {"risk": LABELS[pred], "probabilities": {l: round(float(p), 3) for l, p in zip(LABELS, prob)},
            "alert": bool(alert), "xai": best_xai, "notes": notes, "abnormal_drivers": drivers,
            "other_abnormal": others, "normal_contributors": normal_help, "all_features": findings}


def _val(d):
    """Readable value: Yes/No for the two yes/no features, otherwise number + unit."""
    if d["feature"] in BINARY_NOTES:
        return "Yes" if d["value"] >= 1 else "No"
    return f"{d['value']:g} {d['unit']}"


def _line(d, with_share=True):
    note = f" - {d['note']}" if d["note"] else ""
    share = f"  [{d['share']:.0f}% of the push]" if with_share else ""
    return (f"  - {d['feature']} = {_val(d)}: {d['status'].upper()}{note}\n"
            f"      normal: {d['normal']}{share}")


def show(patient):
    r = assess_patient(patient)
    print(f"Risk: {r['risk']}  {r['probabilities']}")
    print("EARLY WARNING ALERT: " + ("YES -- refer for review" if r["alert"] else "no"))
    print(f"(explained with {r['xai']})")
    if r["abnormal_drivers"]:
        print(f"\nABNORMAL FEATURES THAT LED TO THE {r['risk'].upper()}-RISK RESULT (strongest first):")
        for d in r["abnormal_drivers"]:
            print(_line(d))
    else:
        print(f"\nNo abnormal measurement is pushing this patient toward {r['risk']} risk.")
    if r["normal_contributors"]:
        print("\nNormal values that also supported this result:")
        for d in r["normal_contributors"]:
            print(f"  - {d['feature']} = {_val(d)} (normal: {d['normal']})  [{d['share']:.0f}% of the push]")
    if r["other_abnormal"]:
        print("\nOther abnormal values (present, but with little influence on this prediction):")
        for d in r["other_abnormal"]:
            print(_line(d, with_share=False))
    for n in r["notes"]:
        print(f"  ! {n}")
    print()


def predict_from_input():
    """Type the values in (works in VS Code's input box)."""
    patient = {f: float(input(f"{f} ({UNITS[f]}): ")) for f in FEATURES}
    show(patient)

In [ ]:
# high-risk looking, low-risk looking, borderline
show({"Age": 36, "SystolicBP": 142, "DiastolicBP": 94, "BloodSugar": 12.5, "BodyTemperature": 99.4, "HeartRate": 88,
      "BMI": 32.0, "HemoglobinLevel": 9.6, "GestationalAge": 28, "SmokingAlcoholExposure": 0,
      "PreviousComplications": 1, "WeightGain": 7.0, "PreviousPregnancies": 3, "AntenatalVisitCount": 3})
show({"Age": 23, "SystolicBP": 105, "DiastolicBP": 68, "BloodSugar": 6.8, "BodyTemperature": 98.1, "HeartRate": 70,
      "BMI": 22.5, "HemoglobinLevel": 12.6, "GestationalAge": 22, "SmokingAlcoholExposure": 0,
      "PreviousComplications": 0, "WeightGain": 8.0, "PreviousPregnancies": 1, "AntenatalVisitCount": 8})
show({"Age": 30, "SystolicBP": 120, "DiastolicBP": 78, "BloodSugar": 8.2, "BodyTemperature": 98.6, "HeartRate": 76,
      "BMI": 27.0, "HemoglobinLevel": 11.3, "GestationalAge": 26, "SmokingAlcoholExposure": 0,
      "PreviousComplications": 0, "WeightGain": 9.0, "PreviousPregnancies": 2, "AntenatalVisitCount": 6})

In [ ]:
RUN_INTERACTIVE = False        # set to True and run this cell to enter a patient by hand
if RUN_INTERACTIVE:
    predict_from_input()

## 14. Save the Final Model

The saved pipeline is *Median imputer → Scaler → Classifier* only (SMOTE is a training-time step and is not needed to predict), so it loads with plain scikit-learn. The companion script `predict_maternal_risk.py` loads this file.

In [ ]:
deploy_model = SkPipeline([(n, s) for n, s in best_pipe.steps if n != "smote"])
assert (deploy_model.predict(X_test) == best_pipe.predict(X_test)).all()

artifact = {"model": deploy_model, "model_name": best_name, "features": FEATURES, "classes": LABELS,
            "alert_threshold": ALERT_THRESHOLD, "units": UNITS, "train_range": TRAIN_RANGE.to_dict(),
            "hyperparameters": best_cfg[best_name]["Params"], "smote": best_cfg[best_name]["SMOTE"],
            "test_metrics": {k: float(v) for k, v in evals[best_name]["metrics"].items()},
            "xai_method": best_xai, "xai_comparison": xai_table.round(4).to_dict(),
            "normal_ranges": {"fixed": FIXED_RANGES, "who_contact_weeks": WHO_CONTACT_WEEKS,
                              "iom_total_gain_kg": IOM_TOTAL, "iom_rate_kg_per_week": IOM_RATE}}
with open("finalized_maternal_model.sav", "wb") as f:
    pickle.dump(artifact, f)
print(f"Saved {best_name} pipeline to finalized_maternal_model.sav")

# round-trip check
with open("finalized_maternal_model.sav", "rb") as f:
    loaded = pickle.load(f)
same = (loaded["model"].predict(X_test) == best_pipe.predict(X_test)).all()
print("Reloaded model reproduces the test predictions:", bool(same))

## Conclusion

Four candidates (Logistic Regression, KNN, Random Forest, Gradient Boosting) were run without cross-validation: hyperparameters (and whether to use SMOTE) were tuned on a hold-out validation set, final models were scored once on the test set with Accuracy, Precision, Recall, F1, Sensitivity, Specificity, ROC-AUC and confusion matrices, and the best model was explained with three XAI techniques (SHAP, LIME, Permutation Importance). The techniques were compared on faithfulness, stability and speed, and the best-scoring one now powers a sensitivity-prioritised early-warning predictive system that also reports which abnormal clinical values led to each result.